Q1. Install and load TensorFlow & Keras and print versions

In [ ]:
# Run once (if not installed)
# !pip install -U tensorflow keras

In [ ]:
import tensorflow as tf
import keras

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)

Q2. Load the Wine Quality dataset and explore its dimensions

Dataset link (UCI – red wine):

https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv

In [ ]:
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

df = pd.read_csv(url, sep=";")
df.shape

In [ ]:
df.head()

Q3. Check for null values, identify categorical variables, and encode them

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

There are no categorical columns in this dataset.
So, no encoding is required.

✅ Convert target to binary (required for this assignment)

In [ ]:
df["quality_label"] = (df["quality"] >= 7).astype(int)

Q4. Separate features and target variables

In [ ]:
X = df.drop(["quality", "quality_label"], axis=1)
y = df["quality_label"]

Q5. Train–validation–test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

Q6. Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

Q7. Create at least 2 hidden layers and an output layer

We will use:

2 hidden layers

1 output neuron (binary classification)

Q8. Create Sequential model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

Q9. TensorBoard callback

In [ ]:
from tensorflow.keras.callbacks import TensorBoard
import datetime
import os

log_dir = os.path.join(
    "logs",
    "fit",
    datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
)

tensorboard_cb = TensorBoard(log_dir=log_dir, histogram_freq=1)

Q10. Early Stopping

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping_cb = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

Q11. ModelCheckpoint

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    "best_model.h5",
    monitor="val_loss",
    save_best_only=True
)

Q12. Model summary

In [ ]:
model.summary()

Q13. Loss, optimizer, metric

Loss → binary cross-entropy

Optimizer → Adam

Metric → accuracy

✅ Q14. Compile the model

In [ ]:
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

Q15. Fit the model with callbacks

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[tensorboard_cb, early_stopping_cb, checkpoint_cb]
)

Q16. Get model parameters

In [ ]:
model.count_params()

Q17. Store training history as a DataFrame

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.head()

Q18. Plot the training history

In [ ]:
import matplotlib.pyplot as plt

history_df[["loss", "val_loss"]].plot(figsize=(8,5))
plt.title("Loss")
plt.show()

history_df[["accuracy", "val_accuracy"]].plot(figsize=(8,5))
plt.title("Accuracy")
plt.show()

Q19. Evaluate on test data

In [ ]:
tensorboard --logdir logs/fit